In [ ]:
# Importando as Bibliotecas:
from pathlib import Path
import pandas as pd

# Casamento por Cadastro (mais confiavel que por Nome).
# Usa a base COMPLETA do Ronda (49M registros) ao inves do parquet local reduzido.
pasta = Path('.')
saida = pasta / 'Output_Chapas_Existem_Registros.xlsx'
caminho_ronda_completo = pasta / '..' / '..' / 'Analisando Registros de Colaboradores' / 'data' / 'df_ronda.parquet'

df_chapas = pd.read_excel(pasta / 'base' / 'novaBaseChapa.xlsx')
df_chapas.columns = [str(col).strip() for col in df_chapas.columns]
df_chapas['Cadastro'] = df_chapas['Cadastro'].astype(str).str.strip()
df_chapas['Admissao'] = pd.to_datetime(df_chapas['Admissão'], errors='coerce').dt.normalize()
df_chapas['Demissao'] = pd.to_datetime(df_chapas['Demissão'], errors='coerce').dt.normalize()

cadastros_alvo = set(df_chapas['Cadastro'])
print('Chapas na planilha:', len(df_chapas))
print('Cadastros unicos na planilha:', len(cadastros_alvo))

In [ ]:
# Lendo a base completa do Ronda (apenas colunas necessarias)
df_ronda = pd.read_parquet(caminho_ronda_completo, columns=['cadastro', 'data'])
df_ronda['cadastro'] = df_ronda['cadastro'].astype(str).str.strip()
df_ronda['data'] = df_ronda['data'].dt.normalize()

total_registros_ronda = len(df_ronda)
periodo_inicial = df_ronda['data'].min()
periodo_final = df_ronda['data'].max()
total_dias_possiveis = df_ronda['data'].nunique()
dias_corridos_periodo = (periodo_final - periodo_inicial).days + 1

print('Total de registros no Ronda:', total_registros_ronda)
print('Periodo do Ronda:', periodo_inicial.strftime('%d/%m/%Y'), 'a', periodo_final.strftime('%d/%m/%Y'))
print('Dias com registro no Ronda (universo):', total_dias_possiveis)

In [ ]:
# Filtrando o Ronda apenas para as chapas da planilha
df_ronda_chapas = df_ronda[df_ronda['cadastro'].isin(cadastros_alvo)].copy()
df_ronda_chapas = df_ronda_chapas.drop_duplicates(['cadastro', 'data'])

# Dias com registro por cadastro (geral)
dias_por_cadastro = df_ronda_chapas.groupby('cadastro')['data'].size()
primeiro_por_cadastro = df_ronda_chapas.groupby('cadastro')['data'].min()
ultimo_por_cadastro = df_ronda_chapas.groupby('cadastro')['data'].max()

df_saida = df_chapas.copy()
df_saida['dias_com_registro'] = df_saida['Cadastro'].map(dias_por_cadastro).fillna(0).astype(int)
df_saida['primeiro_registro'] = df_saida['Cadastro'].map(primeiro_por_cadastro)
df_saida['ultimo_registro'] = df_saida['Cadastro'].map(ultimo_por_cadastro)
df_saida['existem_registros'] = df_saida['dias_com_registro'].gt(0).map({True: 'Sim', False: 'Nao'})

# Percentual sobre o universo de dias do Ronda (igual ao calculo antigo)
df_saida['percentual_universo_dias'] = df_saida['dias_com_registro'] / total_dias_possiveis

# Percentual ajustado pela janela de admissao/demissao
# (so conta os dias em que a pessoa estava ativa dentro do periodo do Ronda)
inicio_janela = df_saida['Admissao'].fillna(periodo_inicial).clip(lower=periodo_inicial)
fim_janela = df_saida['Demissao'].fillna(periodo_final).clip(upper=periodo_final)
dias_uteis_individuais = (fim_janela - inicio_janela).dt.days.clip(lower=0) + 1
df_saida['dias_no_periodo_ativo'] = dias_uteis_individuais.fillna(0).astype(int)
df_saida['percentual_periodo_ativo'] = (df_saida['dias_com_registro'] / df_saida['dias_no_periodo_ativo']).replace([float('inf')], 0).fillna(0)

df_saida['primeiro_registro'] = df_saida['primeiro_registro'].dt.strftime('%d/%m/%Y').fillna('-')
df_saida['ultimo_registro'] = df_saida['ultimo_registro'].dt.strftime('%d/%m/%Y').fillna('-')

print('Chapas COM registro:', (df_saida['existem_registros'] == 'Sim').sum())
print('Chapas SEM registro:', (df_saida['existem_registros'] == 'Nao').sum())
display(df_saida.head())

In [ ]:
resumo = pd.DataFrame(
    [
        {'metrica': 'Total de chapas na planilha', 'valor': len(df_saida)},
        {'metrica': 'Cadastros unicos na planilha', 'valor': df_saida['Cadastro'].nunique()},
        {'metrica': 'Chapas com registro no Ronda', 'valor': (df_saida['existem_registros'] == 'Sim').sum()},
        {'metrica': '% de chapas com registro no Ronda', 'valor': ((df_saida['existem_registros'] == 'Sim').mean()) * 100},
        {'metrica': 'Chapas sem registro no Ronda', 'valor': (df_saida['existem_registros'] == 'Nao').sum()},
        {'metrica': '% de chapas sem registro no Ronda', 'valor': ((df_saida['existem_registros'] == 'Nao').mean()) * 100},
        {'metrica': 'Total de registros no Ronda', 'valor': total_registros_ronda},
        {'metrica': 'Total de dias possiveis no Ronda', 'valor': total_dias_possiveis},
        {'metrica': 'Dias corridos no periodo', 'valor': dias_corridos_periodo},
        {'metrica': 'Periodo inicial do Ronda', 'valor': periodo_inicial.strftime('%d/%m/%Y')},
        {'metrica': 'Periodo final do Ronda', 'valor': periodo_final.strftime('%d/%m/%Y')},
        {'metrica': 'Media de dias com registro por chapa', 'valor': df_saida['dias_com_registro'].mean()},
        {'metrica': '% medio do universo de dias por chapa', 'valor': df_saida['percentual_universo_dias'].mean()},
        {'metrica': '% medio do universo de dias entre chapas com registro', 'valor': df_saida.loc[df_saida['existem_registros'] == 'Sim', 'percentual_universo_dias'].mean()},
        {'metrica': '% medio do periodo ativo entre chapas com registro', 'valor': df_saida.loc[df_saida['existem_registros'] == 'Sim', 'percentual_periodo_ativo'].mean()},
    ]
)
display(resumo)

In [ ]:
# Formatando datas para exportar
df_saida_export = df_saida.copy()
df_saida_export['Admissao'] = df_saida_export['Admissao'].dt.strftime('%d/%m/%Y').fillna('-')
df_saida_export['Demissao'] = df_saida_export['Demissao'].dt.strftime('%d/%m/%Y').fillna('-')
df_saida_export = df_saida_export.drop(columns=['Admissão', 'Demissão'], errors='ignore')

with pd.ExcelWriter(saida, engine='openpyxl') as writer:
    df_saida_export.to_excel(writer, sheet_name='output', index=False)
    resumo.to_excel(writer, sheet_name='resumo', index=False)

    wb = writer.book
    ws_output = writer.sheets['output']
    ws_resumo = writer.sheets['resumo']

    for ws in (ws_output, ws_resumo):
        ws.freeze_panes = 'A2'
        for column_cells in ws.columns:
            max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
            ws.column_dimensions[column_cells[0].column_letter].width = min(max(max_len + 2, 12), 55)

    for col_name in ('percentual_universo_dias', 'percentual_periodo_ativo'):
        if col_name in df_saida_export.columns:
            idx = df_saida_export.columns.get_loc(col_name) + 1
            for cell in ws_output.iter_cols(min_col=idx, max_col=idx, min_row=2):
                for item in cell:
                    item.number_format = '0.0000%'

    for row in ws_resumo.iter_rows(min_row=2, max_col=2):
        metrica = row[0].value or ''
        if str(metrica).startswith('%'):
            row[1].number_format = '0.00%'

print(f'Output gerado: {saida.resolve()}')